# Notebook 3 — 3-D Scene Graph Construction

Given detections from each frame, this notebook explains how the system
accumulates them into a persistent 3-D map of objects and builds a
typed relational scene graph.

---

## Prerequisites

| Topic | Why it matters |
|-------|---------------|
| Homogeneous coordinates | Point cloud transforms use 4-D vectors |
| Point clouds | Objects are represented as sets of 3-D points |
| Graph data structures | The scene is a graph of object nodes + relation edges |
| NMS / IoU | The tracker reuses the same overlap concept as detection |

---

## Learning Objectives

1. Understand how depth pixels are back-projected into 3-D point clouds.
2. Understand how the `ObjectTracker3D` matches new detections to existing objects.
3. Run the end-to-end pipeline on a short sequence.
4. Inspect and visualise the resulting scene graph.

## 1 · From Pixels to Point Clouds

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Stage 1: depth image ──────────────────────────────────────────────────────
ax = axes[0]; ax.axis('off')
ax.set_title('① Depth image D  (256×192)', fontsize=10, fontweight='bold')
# Synthetic depth image
np.random.seed(0)
D = np.ones((12,16)) * 2.0
D[4:9, 5:12] = 1.2    # object
D += np.random.randn(12,16)*0.05
im = ax.imshow(D, cmap='plasma', vmin=0.5, vmax=3.0, aspect='auto')
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im, ax=ax, label='depth (m)', shrink=0.7)

# ── Stage 2: binary mask ──────────────────────────────────────────────────────
ax = axes[1]; ax.axis('off')
ax.set_title('② SAM2 mask M  (object pixels only)', fontsize=10, fontweight='bold')
M = np.zeros((12,16), dtype=bool)
M[4:9, 5:12] = True
ax.imshow(M, cmap='Greens', aspect='auto')
ax.set_xticks([]); ax.set_yticks([])

# ── Stage 3: 3-D point cloud ──────────────────────────────────────────────────
ax = axes[2]
ax.set_title('③ 3-D point cloud  P = K⁻¹·D[M]·T_WC', fontsize=10, fontweight='bold')
# Synthetic object point cloud
np.random.seed(42)
n = 80
pts = np.column_stack([
    np.random.uniform(-0.3, 0.3, n),
    np.random.uniform(-0.2, 0.2, n),
    np.random.uniform( 1.0, 1.4, n),
])
ax.scatter(pts[:,0], pts[:,2], c=pts[:,1], cmap='viridis', s=20, alpha=0.8)
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_xlim(-0.8, 0.8); ax.set_ylim(0.5, 2.5)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.suptitle('Back-Projection: Depth × Mask → 3-D Point Cloud', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### The back-projection equation

For each pixel $(u, v)$ where the mask is `True`:

$$
\mathbf{P}^{\text{world}} = T_{WC} \cdot
\begin{pmatrix}
(u - c_x)\, D[v,u] / f_x \\
(v - c_y)\, D[v,u] / f_y \\
D[v,u] \\
1
\end{pmatrix}
$$

The resulting set of 3-D points forms the **point cloud** stored in each `Detection`.

> **Voxel downsampling:** after projection we voxelise at `voxel_size` (default 5 cm)
> to reduce memory and make overlap calculations tractable.

## 2 · The ObjectTracker3D

The tracker maintains a list of **persistent `MapObject`s** across all frames.
When a new frame arrives with $N$ `Detection`s and the tracker holds $M$ existing objects,
it must decide: does detection $i$ correspond to existing object $j$, or is it new?

### Matching criterion

The decision uses a **combined similarity score**:

$$
S(i, j) = w_{\text{geo}} \cdot s_{\text{geo}}(i,j) + w_{\text{sem}} \cdot s_{\text{sem}}(i,j)
$$

| Component | Formula | Intuition |
|-----------|---------|-----------|
| $s_{\text{geo}}$ | voxel overlap fraction | Same physical space |
| $s_{\text{sem}}$ | cosine similarity of SigLIP embeddings | Same semantic category |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── Geometric similarity ──────────────────────────────────────────────────────
ax = axes[0]
np.random.seed(0)
A = np.random.randn(60, 2) * 0.15 + [0, 0]
B = np.random.randn(60, 2) * 0.15 + [0.12, 0.08]
C = np.random.randn(60, 2) * 0.15 + [1.0, 0.5]
ax.scatter(*A.T, c='blue',   alpha=0.4, s=15, label='Existing obj (blue)')
ax.scatter(*B.T, c='orange', alpha=0.4, s=15, label='New det (orange) — HIGH overlap')
ax.scatter(*C.T, c='green',  alpha=0.4, s=15, label='New det (green)  — LOW overlap')
ax.set_title('Geometric Overlap $s_{geo}$', fontsize=10, fontweight='bold')
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.grid(True, alpha=0.3)

# ── Semantic similarity ───────────────────────────────────────────────────────
ax = axes[1]
theta = np.linspace(0, 2*np.pi, 300)
r1 = 1.0
# Two chairs close in embedding space, a table far
chairs_angle = np.array([1.0, 1.15, 4.7])
table_angle  = np.array([3.6])

for a in chairs_angle:
    ax.quiver(0, 0, np.cos(a), np.sin(a), angles='xy', scale_units='xy', scale=1,
              color='blue', alpha=0.7, width=0.015)
for a in table_angle:
    ax.quiver(0, 0, np.cos(a), np.sin(a), angles='xy', scale_units='xy', scale=1,
              color='red',  alpha=0.7, width=0.015)
ax.add_patch(plt.Circle((0,0),1,fill=False,color='grey',ls='--',lw=1))
ax.set_xlim(-1.4,1.4); ax.set_ylim(-1.4,1.4); ax.set_aspect('equal')
ax.text(np.cos(1.0)*1.15, np.sin(1.0)*1.15, 'chair₁', fontsize=8, color='blue')
ax.text(np.cos(1.15)*1.15, np.sin(1.15)*1.15, 'chair₂', fontsize=8, color='blue')
ax.text(np.cos(3.6)*1.15, np.sin(3.6)*1.15, 'table', fontsize=8, color='red')
ax.set_title('Semantic Similarity $s_{sem}$ (cosine)', fontsize=10, fontweight='bold')
ax.set_xlabel('embed dim 1'); ax.set_ylabel('embed dim 2'); ax.grid(True, alpha=0.2)

# ── Aggregated score heatmap ──────────────────────────────────────────────────
ax = axes[2]
n_obj = 5; n_det = 4
scores = np.array([
    [0.85, 0.12, 0.04, 0.02],
    [0.10, 0.78, 0.15, 0.03],
    [0.05, 0.20, 0.72, 0.08],
    [0.02, 0.05, 0.06, 0.81],
    [0.01, 0.02, 0.03, 0.05],
])
im = ax.imshow(scores, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(n_det)); ax.set_xticklabels([f'det {i}' for i in range(n_det)])
ax.set_yticks(range(n_obj)); ax.set_yticklabels([f'obj {i}' for i in range(n_obj)])
for i in range(n_obj):
    for j in range(n_det):
        ax.text(j, i, f'{scores[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='black' if scores[i,j]<0.5 else 'white')
ax.set_title('Aggregated score matrix
(rows=objects, cols=new detections)', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('ObjectTracker3D — Matching Decision', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Merge or create?

After computing the score matrix, for each detection:

- If $\max_j S(i,j) \geq \text{match\_threshold}$ → **merge** into the highest-scoring object
- Otherwise → **create** a new `MapObject`

When merging, the point cloud, feature vector, and crops are fused:

```
obj.pcd  = voxel_downsample(obj.pcd ∪ det.pcd)
obj.clip_ft = (obj.clip_ft * obj.num_views + det.clip_ft) / (obj.num_views + 1)
obj.crops.append(det.crop)
```

> **Periodic merging:** every `K` frames, `prune_and_merge_tracker_objects` runs
> a global Union-Find pass to merge duplicate objects that accumulated over time
> (same class + high geometric overlap but were missed frame-by-frame).

## 3 · Running the Full Pipeline

In [ ]:
import os
from spot_semantic_mapping.configs.loader import cfg
from spot_semantic_mapping.mapping.pipeline import main

DATASET_PATH = '../data/iphone/3578aa5730'
OUTPUT_DIR   = '../outputs/scene_graph_nb03'

# Run on first 100 frames, sampling every 5th (= 20 frames effective)
# Set scene_graph_only=True to skip relation extraction (faster demo)
tracker = main(
    dataset_path=DATASET_PATH,
    rotate=False,
    floor_only=False,
    cfg=cfg,
    ouput_dir=OUTPUT_DIR,
    scene_graph_only=False,
)

print(f'\nPipeline complete.')
print(f'Objects tracked : {len(tracker.objects)}')
print(f'Relation edges  : {len(tracker.edges)}')

In [ ]:
import pandas as pd
import numpy as np

# Summarise each tracked object
rows = []
for obj in tracker.objects:
    cx, cy, cz = obj.bbox_min/2 + obj.bbox_max/2
    rows.append({
        'OID'       : obj.oid,
        'Class'     : obj.class_name,
        'Views'     : obj.num_views,
        'Points'    : len(np.asarray(obj.pcd.points)),
        'Center X'  : round(float(cx), 2),
        'Center Y'  : round(float(cy), 2),
        'Center Z'  : round(float(cz), 2),
        'Caption'   : (obj.caption[:60] + '…') if hasattr(obj,'caption') and obj.caption else '—',
    })

df = pd.DataFrame(rows).sort_values('Views', ascending=False)
print(df.to_string(index=False))

### Multi-view crops

Each `MapObject` accumulates crops from every frame it was visible in. The best 5 are used for captioning.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show multi-view crops for the object with the most views
top_obj = max(tracker.objects, key=lambda o: o.num_views)
crops = top_obj.crops[:8]

n = len(crops)
cols = min(n, 4)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
axes = np.array(axes).reshape(-1) if n > 1 else [axes]

for i, crop in enumerate(crops):
    axes[i].imshow(crop)
    axes[i].set_title(f'View {i+1}', fontsize=9)
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Multi-view crops — "{top_obj.class_name}"  '
             f'({top_obj.num_views} total views)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4 · Scene Graph Structure

The scene graph is a **directed labelled graph**:

```
Nodes  — one per tracked MapObject
         { oid, label, semantic_class, centroid, bbox_min, bbox_max, num_views, caption }

Edges  — spatial relation between two objects
         { src_id, dst_id, relation_type, score }
```

**Relation types** (extracted via VLM voting):

```
left_of · right_of · in_front_of · behind ·
above · below · inside · contains · on_top_of · under ·
next_to · near · overlapping · no_relation
```

In [ ]:
import json, os

graph_path = os.path.join(OUTPUT_DIR, 'scene_graph.json')
with open(graph_path) as f:
    g = json.load(f)

print(f"Nodes : {len(g['nodes'])}")
print(f"Edges : {len(g['edges'])}")
print('\nSample nodes:')
for node in g['nodes'][:4]:
    print(f"  [{node['oid']}] {node['label']:<20s}  "
          f"centroid=({node['centroid'][0]:.2f}, {node['centroid'][1]:.2f}, {node['centroid'][2]:.2f})")
print('\nSample edges:')
for edge in g['edges'][:4]:
    print(f"  [{edge['src_id']}] {edge['src_label']:<15s}  "
          f"─{edge['relation']:^14s}→  [{edge['dst_id']}] {edge['dst_label']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: top-down layout of object centroids ─────────────────────────────────
ax = axes[0]
nodes = g['nodes']
xs = [n['centroid'][0] for n in nodes]
zs = [n['centroid'][2] for n in nodes]

# Colour by semantic class
class_names = sorted(set(n.get('semantic_class','object') for n in nodes))
palette = plt.cm.tab10(np.linspace(0, 1, len(class_names)))
cmap = {c:palette[i] for i,c in enumerate(class_names)}

for n in nodes:
    col = cmap[n.get('semantic_class','object')]
    ax.scatter(n['centroid'][0], n['centroid'][2], c=[col], s=120, zorder=5)
    ax.text(n['centroid'][0]+0.02, n['centroid'][2]+0.02,
            f"[{n['oid']}]{n['label'][:10]}", fontsize=6.5)

for e in g['edges']:
    src = next((n for n in nodes if n['oid']==e['src_id']), None)
    dst = next((n for n in nodes if n['oid']==e['dst_id']), None)
    if src and dst:
        ax.annotate('', xy=(dst['centroid'][0], dst['centroid'][2]),
                    xytext=(src['centroid'][0], src['centroid'][2]),
                    arrowprops=dict(arrowstyle='->', lw=1, color='grey', alpha=0.5))
        mid = ((src['centroid'][0]+dst['centroid'][0])/2,
               (src['centroid'][2]+dst['centroid'][2])/2)
        ax.text(*mid, e['relation'], fontsize=6, color='darkblue', alpha=0.7)

ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title('Top-down Scene Graph (XZ plane)', fontsize=11, fontweight='bold')
legend_patches = [mpatches.Patch(color=cmap[c], label=c) for c in class_names]
ax.legend(handles=legend_patches, fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# ── Right: NetworkX graph ──────────────────────────────────────────────────────
ax = axes[1]
G = nx.DiGraph()
for n in nodes:
    G.add_node(n['oid'], label=n['label'][:10])
for e in g['edges']:
    G.add_edge(e['src_id'], e['dst_id'], relation=e['relation'])

pos = nx.spring_layout(G, seed=42, k=2)
node_colors = [cmap.get(
    next((n.get('semantic_class','object') for n in nodes if n['oid']==v), 'object'),
    (0.5,0.5,0.5,1)
) for v in G.nodes()]

nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=400, alpha=0.9)
nx.draw_networkx_labels(G, pos, ax=ax,
    labels={n['oid']: f"{n['oid']}:{n['label'][:8]}" for n in nodes},
    font_size=6)
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle='-|>',
    edge_color='grey', alpha=0.6, connectionstyle='arc3,rad=0.1')
edge_labels = {(e['src_id'],e['dst_id']): e['relation'] for e in g['edges']}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=6)

ax.set_title('Scene Graph (NetworkX)', fontsize=11, fontweight='bold')
ax.axis('off')

plt.suptitle('Scene Graph Visualisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

| Concept | Implementation |
|---------|---------------|
| Back-projection | `K⁻¹ · depth · T_WC` vectorised over masked pixels |
| Object matching | Geo + semantic score, threshold → merge or create |
| Multi-view fusion | Running-average clip features + crop list |
| Global merging | Union-Find with combined score after every K frames |
| Relations | VLM voting over KNN graph, majority label wins |
| Export | `scene_graph.json` (nodes + edges) + `semantic_cloud.ply` |

**Next:** Notebook 4 uses this scene graph database for visual place recognition.